In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

pd.set_option('display.max_columns', None)
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from DATA.TOOLS.fetchPlayersStats import FetchPlayersStats
from DATA.TOOLS.fetchTeamStats import *

# feature_path = os.path.abspath(r"C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor")
feature_path = os.path.abspath('/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor')
# Add the parent directory to sys.path
if feature_path not in sys.path:
    sys.path.append(feature_path)

from FEATURE_ENGINEERING.features import *
from DATA.TOOLS.playerPositions import *

Features I plan on adding in the future

Player-specific usage and scoring data

- /BoxScoreScoringV2: Breaks down how a player scores (paint, midrange, 3s, free throws). *

Shot quality and location data

- /ShotChartDetail: Individual shot attempts, zones, and frequencies.

- /LeagueDashPlayerShotLocations: Aggregated shot location tendencies.

- /LeagueDashPlayerPtShot: Breaks down shooting by play type and situation.

Opponent and matchup data

- /LeagueDashPtDefend: How defenders contest shots and limit scoring. *

- /LeagueDashTeamStats with defense filters: Opponent’s defensive efficiency.

- /BoxScoreMatchupsV3: Player-vs-player defensive assignments.

Game context and pace

- /ScoreboardV2 or /PlayByPlayV2: For back-to-backs, rest, or pace indicators.

- /TeamDashboardByGeneralSplits: Team-level pace, offensive rating, and context.


In [ ]:
from nba_api.stats.endpoints import boxscorescoringv3
from nba_api.stats.endpoints import boxscorescoringv3
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import random
from requests.exceptions import ReadTimeout, ConnectionError

def fetch_game_data(gameId, sleep_time=1.0):
    """Fetch data for a single game with error handling"""
    try:
        # Add sleep time to avoid rate limiting
        time.sleep(sleep_time)
        
        boxscore = boxscorescoringv3.BoxScoreScoringV3(
            game_id=f'00{gameId}',
            timeout=60
        )
        data_frames = boxscore.get_data_frames()
        
        # Check if we got valid data
        if data_frames and len(data_frames) > 0 and not data_frames[0].empty:
            return data_frames[0], gameId, None
        else:
            return None, gameId, "No data available"
            
    except ReadTimeout:
        return None, gameId, "Timeout"
    except Exception as e:
        return None, gameId, str(e)

# Get unique game IDs
gameIds = s19_regular['GAME_ID'].unique()
games = []
failed_games = []

print(f"Processing {len(gameIds)} games using ThreadPoolExecutor...")

# Use ThreadPoolExecutor with a reasonable number of workers
max_workers = 3  # Start with 3 to avoid overwhelming the API
sleep_time = 1.0  # Sleep time between requests

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    # Submit all tasks
    future_to_game = {
        executor.submit(fetch_game_data, gameId, sleep_time): gameId 
        for gameId in gameIds
    }
    
    # Process completed tasks
    for i, future in enumerate(as_completed(future_to_game), 1):
        gameId = future_to_game[future]
        
        try:
            result, game_id, error = future.result()
            
            if result is not None:
                games.append(result)
                print(f"✓ Game {gameId} completed ({i}/{len(gameIds)})")
            else:
                failed_games.append((gameId, error))
                print(f"✗ Game {gameId} failed: {error} ({i}/{len(gameIds)})")
                
        except Exception as e:
            failed_games.append((gameId, str(e)))
            print(f"✗ Game {gameId} exception: {str(e)} ({i}/{len(gameIds)})")

print(f"\nCompleted!")
print(f"Successfully processed: {len(games)} games")
print(f"Failed games: {len(failed_games)}")

if failed_games:
    print(f"Failed games: {failed_games[:10]}...")

if games:
    games_df = pd.concat(games, ignore_index=True)
    print(f"\nFinal dataset shape: {games_df.shape}")
    games_df.head()
else:
    print("No games were successfully processed!")

Processing 1230 games using ThreadPoolExecutor...
✓ Game 21800002 completed (1/1230)
✓ Game 21800007 completed (2/1230)
✓ Game 21800001 completed (3/1230)
✓ Game 21800012 completed (4/1230)
✓ Game 21800008 completed (5/1230)
✓ Game 21800009 completed (6/1230)
✓ Game 21800003 completed (7/1230)
✓ Game 21800004 completed (8/1230)
✓ Game 21800006 completed (9/1230)
✓ Game 21800013 completed (10/1230)
✓ Game 21800005 completed (11/1230)
✓ Game 21800011 completed (12/1230)
✓ Game 21800010 completed (13/1230)
✓ Game 21800014 completed (14/1230)
✓ Game 21800016 completed (15/1230)
✓ Game 21800015 completed (16/1230)
✓ Game 21800020 completed (17/1230)
✓ Game 21800019 completed (18/1230)
✓ Game 21800022 completed (19/1230)
✓ Game 21800024 completed (20/1230)
✓ Game 21800021 completed (21/1230)
✓ Game 21800025 completed (22/1230)
✓ Game 21800023 completed (23/1230)
✓ Game 21800018 completed (24/1230)
✓ Game 21800017 completed (25/1230)
✓ Game 21800028 completed (26/1230)
✓ Game 21800031 complet

## Fetches Player Gamelogs

In [2]:
# nba = FetchPlayersStats()
# data = nba.getCompleteStats(
#     season='2018-19', 
#     season_type='Regular Season', 
#     sleep_time=1.5, 
#     max_workers=5,
#     batch_limit=100,
#     complete_cache_file='../DATA/CSV_FILES/REGULAR_DATA/S19.csv'
# )
# data.head()

In [3]:
pd.set_option('display.max_columns', None)
def convert_min_to_float(min_str):
    try:
        if isinstance(min_str, str) and ":" in min_str:
            minutes, seconds = map(int, min_str.split(":"))
            total_minutes = minutes + seconds / 60
            return round(total_minutes, 2)
        elif isinstance(min_str, (int, float)):
            return float(min_str)
        else:
            return 0
    except:
        return 0
    

s19_regular = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/S19.csv')
s19_regular['IS_PLAYOFF'] = 0
s19_regular['MIN'] = s19_regular['MIN'].apply(convert_min_to_float)

s20_regular = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/S20.csv')
s20_regular['IS_PLAYOFF'] = 0
s20_regular['MIN'] = s20_regular['MIN'].apply(convert_min_to_float)

s21_playoffs = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/P21.csv')
s21_playoffs['IS_PLAYOFF'] = 1
s21_playoffs['MIN'] = s21_playoffs['MIN'].apply(convert_min_to_float)
s21_regular = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/S21.csv')
s21_regular['IS_PLAYOFF'] = 0
s21_regular['MIN'] = s21_regular['MIN'].apply(convert_min_to_float)
s21 = pd.concat([s21_regular, s21_playoffs])

s22_playoffs = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/P22.csv')
s22_playoffs['IS_PLAYOFF'] = 1
s22_playoffs['MIN'] = s22_playoffs['MIN'].apply(convert_min_to_float)
s22_regular = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/S22.csv')
s22_regular['IS_PLAYOFF'] = 0
s22_regular['MIN'] = s22_regular['MIN'].apply(convert_min_to_float)
s22 = pd.concat([s22_regular, s22_playoffs])

s23_playoffs = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/P23.csv')
s23_playoffs['IS_PLAYOFF'] = 1
s23_playoffs['MIN'] = s23_playoffs['MIN'].apply(convert_min_to_float)
s23_regular = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/S23.csv')
s23_regular['IS_PLAYOFF'] = 0
s23_regular['MIN'] = s23_regular['MIN'].apply(convert_min_to_float)
s23 = pd.concat([s23_regular, s23_playoffs])

s24_playoffs = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/P24.csv')
s24_playoffs['IS_PLAYOFF'] = 1
s24_playoffs['MIN'] = s24_playoffs['MIN'].apply(convert_min_to_float)
s24_regular = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/S24.csv')
s24_regular['IS_PLAYOFF'] = 0
s24_regular['MIN'] = s24_regular['MIN'].apply(convert_min_to_float)
s24 = pd.concat([s24_regular, s24_playoffs])

s25_playoffs = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/P25.csv')
s25_playoffs['IS_PLAYOFF'] = 1
s25_playoffs['MIN'] = s25_playoffs['MIN'].apply(convert_min_to_float)
s25_regular = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/S25.csv')
s25_regular['IS_PLAYOFF'] = 0
s25_regular['MIN'] = s25_regular['MIN'].apply(convert_min_to_float)
s25 = pd.concat([s25_regular, s25_playoffs])
s19_regular

,Unnamed: 0.1,Unnamed: 0,PLAYER_NAME,PLAYER_ID,MATCHUP,TEAM_ABBREVIATION,TEAM_ID,OPP_ABBREVIATION,HOME_GAME,GAME_ID,GAME_DATE,WL,PTS,AST,REB,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,STL,BLK,TOV,PLUS_MINUS,FANTASY_PTS,POINT_PER_SHOT,EFG,START_POSITION,COMMENT,OFF_RATING,E_OFF_RATING,DEF_RATING,E_DEF_RATING,NET_RATING,OREB_PCT,DREB_PCT,REB_PCT,AST_PCT,EFG_PCT,AST_TOV,USG_PCT,TS_PCT,E_PACE,PACE,PIE,POSS,PACE_PER40,E_USG_PCT,MIN,SPD,DIST,ORBC,DRBC,RBC,TCHS,SAST,FTAST,PASS,CFGM,CFGA,CFG_PCT,UFGM,UFGA,UFG_PCT,DFGM,DFGA,DFG_PCT,PTS_OFF_TOV,PTS_2ND_CHANCE,PTS_FB,PTS_PAINT,OPP_PTS_OFF_TOV,OPP_PTS_2ND_CHANCE,OPP_PTS_FB,OPP_PTS_PAINT,BLKA,PF,PFD,IS_PLAYOFF,TEAM_SEASON_ID,TEAM_NAME,TEAM_GAME_DATE,TEAM_MATCHUP,TEAM_WL,TEAM_MIN,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_STL,TEAM_BLK,TEAM_TOV,TEAM_PF,TEAM_PTS,TEAM_PLUS_MINUS,VIDEO_AVAILABLE,TEAM_PACE,GAME_PACE,OPP_PACE,OPP_TEAM_ID,TEAM_OFF_RATING,TEAM_DEF_RATING,OPP_DEF_RATING,OPP_OFF_RATING,OPP_PTS,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_REB,OPP_AST,OPP_STL,OPP_BLK,OPP_TOV,whos_favored,spread,total,team_is_favored,team_spread
0,0,0,Daniel Theis,1628464,BOS vs. PHI,BOS,1610612738,PHI,1,21800001,2018-10-16,W,0,0,1,0,0,NaN,0,0,NaN,0,0,NaN,0,1,0,0,0,3,1.2,0.000,NaN,NaN,NaN,111.1,111.1,70.0,70.0,41.1,0.000,0.200,0.111,0.000,0.000,0.00,0.000,0.000,110.32,110.32,0.000,9.0,91.94,0.000,4.13,4.23,0.32,2,3,4,3,0,0,3,0,0,0.00,0,0,0.000,1,1,1.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,1.0,0.0,0,22018,Boston Celtics,2018-10-16,BOS vs. PHI,W,240,42,97,0.433,11,37,0.297,10,14,0.714,12,43,55,21,7,5,15,20,105,18,1,106.6,106.6,106.6,1610612755,98.9,82.0,98.0,81.2,87.0,34.0,87.0,0.391,47.0,18.0,8.0,5.0,16.0,home,4.5,211.5,True,4.5
1,1,1,Klay Thompson,202691,GSW vs. OKC,GSW,1610612744,OKC,1,21800002,2018-10-16,W,14,0,4,5,20,0.250,1,8,0.125,3,3,1.000,1,3,0,0,2,2,16.8,0.657,0.275000,G,NaN,106.7,106.7,102.6,102.6,4.0,0.026,0.067,0.048,0.000,0.275,0.00,0.250,0.328,103.59,103.59,-0.021,75.0,86.33,0.250,34.98,4.35,2.72,3,5,8,45,0,0,22,0,6,0.00,5,14,0.357,2,2,1.000,3.0,2.0,0.0,4.0,18.0,14.0,8.0,26.0,2.0,3.0,3.0,0,22018,Golden State Warriors,2018-10-16,GSW vs. OKC,W,240,42,95,0.442,7,26,0.269,17,18,0.944,17,41,58,28,7,7,21,29,108,8,1,106.6,106.6,106.6,1610612760,101.0,93.5,101.6,94.1,100.0,33.0,91.0,0.363,45.0,21.0,12.0,6.0,15.0,home,12.0,220.5,True,12.0
2,2,2,Draymond Green,203110,GSW vs. OKC,GSW,1610612744,OKC,1,21800002,2018-10-16,W,2,5,13,1,6,0.167,0,1,0.000,0,0,NaN,1,12,3,0,6,2,28.1,0.333,0.166667,F,NaN,100.0,100.0,94.8,94.8,5.2,0.033,0.279,0.178,0.185,0.167,0.83,0.143,0.167,111.39,111.39,0.071,75.0,92.82,0.143,32.75,4.05,2.37,6,20,22,79,1,1,62,1,4,0.25,0,2,0.000,2,3,0.667,0.0,0.0,0.0,2.0,14.0,10.0,8.0,20.0,1.0,3.0,1.0,0,22018,Golden State Warriors,2018-10-16,GSW vs. OKC,W,240,42,95,0.442,7,26,0.269,17,18,0.944,17,41,58,28,7,7,21,29,108,8,1,106.6,106.6,106.6,1610612760,101.0,93.5,101.6,94.1,100.0,33.0,91.0,0.363,45.0,21.0,12.0,6.0,15.0,home,12.0,220.5,True,12.0
3,3,3,Alfonzo McKinnie,1628035,GSW vs. OKC,GSW,1610612744,OKC,1,21800002,2018-10-16,W,0,0,0,0,1,0.000,0,1,0.000,0,0,NaN,0,0,0,0,0,-1,0.0,0.000,0.000000,NaN,NaN,80.0,80.0,83.3,83.3,-3.3,0.000,0.000,0.000,0.000,0.000,0.00,0.200,0.000,112.34,112.34,-0.143,5.0,93.62,0.200,2.35,4.38,0.18,0,0,0,2,0,0,1,0,0,0.00,0,1,0.000,0,0,0.000,0.0,0.0,0.0,0.0,3.0,0.0,0.0,2.0,0.0,0.0,0.0,0,22018,Golden State Warriors,2018-10-16,GSW vs. OKC,W,240,42,95,0.442,7,26,0.269,17,18,0.944,17,41,58,28,7,7,21,29,108,8,1,106.6,106.6,106.6,1610612760,101.0,93.5,101.6,94.1,100.0,33.0,91.0,0.363,45.0,21.0,12.0,6.0,15.0,home,12.0,220.5,True,12.0
4,4,4,JJ Redick,200755,PHI @ BOS,PHI,1610612755,BOS,0,21800001,2018-10-16,L,16,1,2,7,17,0.412,2,8,0.250,0,0,NaN,0,2,0,0,0,-10,19.9,0.941,0.470588,NaN,NaN,82.9,82.9,100.0,100.0,-17.1,0.000,0.057,0.029,0.063,0.471,0.00,0.224,0.471,111.27,111.27,0.080,70.0,92.72,0.224,29.77,4.49,2.42,0,4,4,39,1,0,20,1,2,0.50,6,15,0.400,1,2,0.500,0.0,0.0,0.0,2.0,10.0,5.0,6.0,

## Merge team data into player logs

In [4]:
# # Drop all team-related columns to re-merge with correct pace calculations
# data = s19_regular

# columns_to_drop = [
#     'Unnamed: 0.3', 'Unnamed: 0.2', 'Unnamed: 0.1', 'Unnamed: 0',
#     # Pace columns (incorrect calculations)
#     'TEAM_PACE', 'GAME_PACE', 'OPP_PACE',
    
#     # Rating columns (may need recalculation with correct pace)
#     'TEAM_OFF_RATING', 'TEAM_DEF_RATING',
#     'OPP_OFF_RATING', 'OPP_DEF_RATING',
    
#     # Other team stats that came from the merge
#     'TEAM_PTS', 'TEAM_FGM', 'TEAM_FGA', 'TEAM_FG_PCT',
#     'TEAM_FG3M', 'TEAM_FG3A', 'TEAM_FG3_PCT',
#     'TEAM_FTM', 'TEAM_FTA', 'TEAM_FT_PCT', 
#     'TEAM_OREB', 'TEAM_DREB', 'TEAM_REB',
#     'TEAM_AST', 'TEAM_STL', 'TEAM_BLK', 'TEAM_TOV',
#     'TEAM_PF', 'TEAM_PLUS_MINUS',
    
#     # Opponent stats
#     'OPP_TEAM_ID', 'OPP_PTS', 'OPP_FGM', 'OPP_FGA', 'OPP_FG_PCT',
#     'OPP_REB', 'OPP_AST', 'OPP_STL', 'OPP_BLK', 'OPP_TOV'
    
#     'TEAM_SEASON_ID_x',	'OPP_TOV_x', 'TEAM_SEASON_ID_y', 'OPP_TOV_y', 'TEAM_SEASON_ID', 'TEAM_NAME', 'TEAM_GAME_DATE', 'TEAM_MATCHUP', 'TEAM_WL', 'TEAM_MIN', 'VIDEO_AVAILABLE'
# ]

# # Drop columns that exist in the dataframe
# existing_cols = [col for col in columns_to_drop if col in data.columns]
# data = data.drop(columns=existing_cols)

# teamlogs = mergeTeamtoPlayer(data, season='2018-19', season_type='Regular Season')
# teamlogs


In [5]:
# teamlogs.to_csv('../DATA/CSV_FILES/REGULAR_DATA/S19.csv')
# teamlogs

## Assign features for regular season data

In [6]:
star_players_by_year = {
    2019:[ "Giannis Antetokounmpo",
    "LeBron James",
    "Anthony Davis",
    "James Harden",
    "Luka Dončić",
    "Kawhi Leonard",
    "Pascal Siakam",
    "Nikola Jokić",
    "Damian Lillard",
    "Chris Paul",
    "Jayson Tatum",
    "Jimmy Butler",
    "Rudy Gobert",
    "Ben Simmons",
    "Russell Westbrook"],
    2020:[
    "Giannis Antetokounmpo",
    "Kawhi Leonard", 
    "Nikola Jokić",
    "Stephen Curry",
    "Luka Dončić",
    "Julius Randle",
    "LeBron James",
    "Joel Embiid",
    "Chris Paul",
    "Damian Lillard",
    "Jimmy Butler",
    "Paul George",
    "Rudy Gobert",
    "Bradley Beal",
    "Kyrie Irving"
    ],
    2021: [
        "Giannis Antetokounmpo", "Kawhi Leonard", "Nikola Jokić", "Stephen Curry", "Luka Dončić",
        "Julius Randle", "LeBron James", "Joel Embiid", "Damian Lillard", "Chris Paul",
        "Jimmy Butler", "Paul George", "Rudy Gobert", "Bradley Beal", "Kyrie Irving",
        "Devin Booker", "Mike Conley", "James Harden", "Zach LaVine", "Donovan Mitchell",
        "Nikola Vucevic", "Anthony Davis"
    ],
    2022: [
        "Giannis Antetokounmpo", "Luka Dončić", "Jayson Tatum", "Nikola Jokić", "Devin Booker",
        "Ja Morant", "Stephen Curry", "DeMar DeRozan", "Kevin Durant", "Joel Embiid",
        "LeBron James", "Chris Paul", "Trae Young", "Pascal Siakam", "Karl-Anthony Towns",
        "Andrew Wiggins", "Donovan Mitchell", "Rudy Gobert", "Zach LaVine", "Khris Middleton",
        "Jimmy Butler", "Darius Garland", "Fred VanVleet", "LaMelo Ball"
    ],
    2023: [
        "Giannis Antetokounmpo", "Jayson Tatum", "Joel Embiid", "Shai Gilgeous-Alexander", "Luka Dončić",
        "Jaylen Brown", "Jimmy Butler", "Nikola Jokić", "Stephen Curry", "Donovan Mitchell",
        "LeBron James", "Julius Randle", "Domantas Sabonis", "De'Aaron Fox", "Damian Lillard",
        "Kyrie Irving", "Zion Williamson", "Kevin Durant", "Ja Morant", "DeMar DeRozan",
        "Tyrese Haliburton", "Jrue Holiday", "Bam Adebayo", "Jaren Jackson Jr.", "Paul George",
        "Pascal Siakam", "Anthony Edwards"
    ],
    2024: [
        "Shai Gilgeous-Alexander", "Luka Dončić", "Jayson Tatum", "Giannis Antetokounmpo", "Nikola Jokić",
        "Jalen Brunson", "Anthony Edwards", "Kawhi Leonard", "Kevin Durant", "Anthony Davis",
        "Stephen Curry", "Devin Booker", "LeBron James", "Domantas Sabonis", "Bam Adebayo",
        "Tyrese Haliburton", "Damian Lillard", "Karl-Anthony Towns", "Jaylen Brown",
        "Trae Young", "Paolo Banchero", "Scottie Barnes", 'Joel Embiid'
    ],
    2025: [
        'Shai Gilgeous-Alexander', 'Nikola Jokić', 'Giannis Antetokounmpo', 'Jayson Tatum', 'Donovan Mitchell',
        'Anthony Edwards', 'LeBron James', 'Stephen Curry', 'Evan Mobley', 'Jalen Brunson',
        'Cade Cunningham', 'Karl-Anthony Towns', 'Tyrese Haliburton', 'Jalen Williams', 'James Harden',
        'Darius Garland', 'Damian Lillard', 'Anthony Davis', 'Kyrie Irving', 'Jaylen Brown', 'Tyler Herro', 'Jaren Jackson Jr.', 
        'Pascal Siakam', 'Victor Wembanyama', 'Alperen Sengun', 'Trae Young', 'LaMelo Ball', 'Devin Booker', 'Joel Embiid', 'Luka Dončić'
]}

In [7]:
def process_season_features(season_df, prop_type, year, star_players):
    df = season_df.copy()
    df = sort_data_for_features(df)
    df['STARTING'] = df['START_POSITION'].apply(lambda x: 1 if x in ['G','F','C'] else 0)
    # df['BLOWOUT_RISK'] = (abs(df['spread']) >= 10).astype(int)
    df['TEAM_IMPLIED_PTS_FAV'] = ((df['total'] + df['team_spread']) / 2).round(1)
    df['TEAM_IMPLIED_PTS_UND'] = ((df['total'] - df['team_spread']) / 2).round(1)
    
    # Add position data
    cache_file = os.path.join(feature_path, 'DATA', 'TOOLS', 'playerInfo.csv')
    df = assign_position_with_cache(
        df, 
        cache_file=cache_file,
        max_workers=4, 
        delay_between_requests=1.5
    )
    # Basic features to track rest and travel
    df = add_rest_day_features(df)
    
    # Prop-specific features
    df = rollingAverages(df, player_id_col='PLAYER_ID', date_col='GAME_DATE', windows=[3,5,7])
    df = statAgainstTeam(df, player_id_col='PLAYER_ID', opp_col='OPP_ABBREVIATION', stat_line=prop_type)
    df = HomeAwayAverages(df, player_id_col='PLAYER_ID', date_col='GAME_DATE')
    df = getPlayerAvgToDateVectorized(df)
    df = addLagFeatures(df, stat_lines=['PTS', 'MIN', 'FGA', 'FTA', 'FG3A', 'FG_PCT', 'FG3_PCT', 'FT_PCT', 'USG_PCT', 'TS_PCT',
                        'EFG_PCT','POSS', 'TCHS','AST', 'REB', 'TOV'])
    # df = add_volatility_features(df, windows=[3,5,7,15,40])
    # df = add_performance_volatility_categories(df)
    # df = add_recent_form_volatility(df)
    df = process_star_players_data(df, star_players, min_minutes=20)
    df = add_performance_without_stars_columns(df, min_games=1)
    df = teamUsualStarters(df)
    df = oppTeamUsualStarters(df)
    # df = add_all_defensive_features(df, all_defensive_players, year)
    df = teamContext(df)
    df = assign_opponent_team_stats_dict(df)
    df = expectedPace(df)
    
    # Clean up any unwanted columns
    if 'Unnamed: 0' in df.columns:
        df.drop(columns=['Unnamed: 0'], inplace=True)
    if 'Unnamed: 0.1' in df.columns:
        df.drop(columns=['Unnamed: 0.1'], inplace=True)
    return df

prop_types = ['PTS']
data = [s19_regular,s20_regular, s21_regular,s22_regular,s23_regular,s24_regular,s25_regular]
seasons = [2019,2020,2021,2022,2023,2024,2025]

# Pre-define output directory once
output_dir = os.path.join(feature_path, 'DATA', 'CSV_FILES', 'TRAIN_DATA')
os.makedirs(output_dir, exist_ok=True)

# Process each season sequentially but with optimized operations
for season_data, year in zip(data, seasons):
    print(f"Processing year {year}...")
    
    # Process features
    processed_data = process_season_features(
        season_data, 
        prop_type='PTS',
        year=year,
        star_players=set(star_players_by_year[year]),
        # all_defensive_players=set(all_defensive_players[year])
    )
    
    # Save file
    output_path = os.path.join(output_dir, f'PTS_TRAIN_{str(year)[-2:]}.csv')
    processed_data.to_csv(output_path)
    print(f"Completed {year}")

Processing year 2019...
Loading position cache...
Loaded 992 players from cache
Found 530 unique players, 159 need to be fetched
Fetching 159 new players using 4 threads...
Fetched PLAYER_ID 1717... (1/159)
Fetched PLAYER_ID 1713... (2/159)
Fetched PLAYER_ID 2037... (3/159)
Fetched PLAYER_ID 2199... (4/159)
Fetched PLAYER_ID 2200... (5/159)
Fetched PLAYER_ID 2225... (6/159)
Fetched PLAYER_ID 2403... (7/159)
Fetched PLAYER_ID 2548... (8/159)
Fetched PLAYER_ID 2585... (9/159)
Fetched PLAYER_ID 2594... (10/159)
Fetched PLAYER_ID 2733... (11/159)
Fetched PLAYER_ID 2734... (12/159)
Fetched PLAYER_ID 2736... (13/159)
Fetched PLAYER_ID 101107... (14/159)
Fetched PLAYER_ID 2747... (15/159)
Fetched PLAYER_ID 101106... (16/159)
Fetched PLAYER_ID 101109... (17/159)
Fetched PLAYER_ID 101112... (18/159)
Fetched PLAYER_ID 101123... (19/159)
Fetched PLAYER_ID 101133... (20/159)
Fetched PLAYER_ID 101161... (21/159)
Fetched PLAYER_ID 101162... (22/159)
Fetched PLAYER_ID 101181... (23/159)
Fetched PLAYE

/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/FEATURE_ENGINEERING/features.py:140: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[rolling_col_name] = rolling_avg.fillna(df[expanding_col])
/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/FEATURE_ENGINEERING/features.py:140: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[rolling_col_name] = rolling_avg.fillna(df[expanding_col])
/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/FEATURE_ENGINEERING/features.py:140: PerformanceWarning: DataFrame

Completed 2019
Processing year 2020...
Loading position cache...
Loaded 1151 players from cache
Found 529 unique players, 24 need to be fetched
Fetching 24 new players using 4 threads...
Fetched PLAYER_ID 1626187... (1/24)
Fetched PLAYER_ID 1627784... (2/24)
Fetched PLAYER_ID 202951... (3/24)
Fetched PLAYER_ID 203705... (4/24)
Fetched PLAYER_ID 1628430... (5/24)
Fetched PLAYER_ID 1627982... (6/24)
Fetched PLAYER_ID 1628450... (7/24)
Fetched PLAYER_ID 1628499... (8/24)
Fetched PLAYER_ID 1628987... (9/24)
Fetched PLAYER_ID 1628985... (10/24)
Fetched PLAYER_ID 1629044... (11/24)
Fetched PLAYER_ID 1629065... (12/24)
Fetched PLAYER_ID 1629598... (13/24)
Fetched PLAYER_ID 1629608... (14/24)
Fetched PLAYER_ID 1629625... (15/24)
Fetched PLAYER_ID 1629621... (16/24)
Fetched PLAYER_ID 1629668... (17/24)
Fetched PLAYER_ID 1629724... (18/24)
Fetched PLAYER_ID 1629729... (19/24)
Fetched PLAYER_ID 1629734... (20/24)
Fetched PLAYER_ID 1629739... (21/24)
Fetched PLAYER_ID 1629741... (22/24)
Fetched PL

/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/FEATURE_ENGINEERING/features.py:140: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[rolling_col_name] = rolling_avg.fillna(df[expanding_col])
/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/FEATURE_ENGINEERING/features.py:140: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[rolling_col_name] = rolling_avg.fillna(df[expanding_col])
/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/FEATURE_ENGINEERING/features.py:140: PerformanceWarning: DataFrame

Completed 2020
Processing year 2021...
Loading position cache...
Loaded 1175 players from cache
Found 540 unique players, 0 need to be fetched
Saving updated cache...
Cache saved with 1175 players
Applying positions to dataset...
Creating position flags...
Position assignment completed!


...

Completed 2023
Processing year 2024...
Loading position cache...
Loaded 1175 players from cache
Found 572 unique players, 0 need to be fetched
Saving updated cache...
Cache saved with 1175 players
Applying positions to dataset...
Creating position flags...
Position assignment completed!


/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/FEATURE_ENGINEERING/features.py:140: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[rolling_col_name] = rolling_avg.fillna(df[expanding_col])
/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/FEATURE_ENGINEERING/features.py:140: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[rolling_col_name] = rolling_avg.fillna(df[expanding_col])
/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/FEATURE_ENGINEERING/features.py:140: PerformanceWarning: DataFrame

Completed 2024
Processing year 2025...
Loading position cache...
Loaded 1175 players from cache
Found 569 unique players, 0 need to be fetched
Saving updated cache...
Cache saved with 1175 players
Applying positions to dataset...
Creating position flags...
Position assignment completed!


/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/FEATURE_ENGINEERING/features.py:140: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[rolling_col_name] = rolling_avg.fillna(df[expanding_col])
/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/FEATURE_ENGINEERING/features.py:140: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[rolling_col_name] = rolling_avg.fillna(df[expanding_col])
/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/FEATURE_ENGINEERING/features.py:140: PerformanceWarning: DataFrame

Completed 2025


In [8]:
# df = s21

# df['STARTING'] = df['START_POSITION'].apply(lambda x: 1 if x in ['G','F','C'] else 0)
# df['team_is_favored'] = df['team_is_favored'].astype(int)
# df['whos_favored'] = df['whos_favored'].apply(lambda x: 1 if x == 'home' else 0)
# df['BLOWOUT_RISK'] = (abs(df['spread']) >= 10).astype(int)

# # Add position data
# cache_file = os.path.join(feature_path, 'DATA', 'TOOLS', 'playerInfo.csv')
# df = assign_position_with_cache(
#     df, 
#     cache_file=cache_file,
#     max_workers=4, 
#     delay_between_requests=1.5
# )
# df = add_rest_day_features(df)
# df = add_minutes_trend_features(df)
# df = add_lineup_cohesion(df)
# df = add_rotation_stability(df)
# df = add_usage_shift(df)
# df = MINrollingAverages(df, player_id_col='PLAYER_ID', date_col='GAME_DATE', windows=[3,5,7])
# df = MINLagFeatures(df, player_id_col='PLAYER_ID', date_col='GAME_DATE', stat_line='MIN')
# df = getPlayerMINAvgToDate(df, player_id_col='PLAYER_ID', date_col='GAME_DATE')
# df = MINHomeAwayAverages(df, player_id_col='PLAYER_ID', date_col='GAME_DATE')
# df = MINAgainstTeam(df, player_id_col='PLAYER_ID', opp_col='OPP_ABBREVIATION', stat_line='MIN')
# df = assign_opponent_team_stats_dict(df)
# df = teamContext(df)
# df.to_csv('../DATA/CSV_FILES/TRAIN_DATA/MIN_TRAIN_21.csv', index=False)

In [9]:
def merge_betting_data(player_df, betting_df, team_dict):
    """
    Merge betting data (spread, total, who's favored) into player dataset
    """
    df = player_df.copy()
    odds = betting_df.copy()
    df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
    odds['date'] = pd.to_datetime(odds['date'])
    
    # Convert betting data team abbreviations to uppercase using team_dict
    odds['away_upper'] = odds['away'].map(team_dict)
    odds['home_upper'] = odds['home'].map(team_dict)
    
    # First, create a unique identifier for each game in odds data
    odds['game_key_home'] = odds['date'].astype(str) + '_' + odds['home_upper'] + '_' + odds['away_upper']
    odds['game_key_away'] = odds['date'].astype(str) + '_' + odds['away_upper'] + '_' + odds['home_upper']
    
    df['game_key'] = df['GAME_DATE'].astype(str) + '_' + df['TEAM_ABBREVIATION'] + '_' + df['OPP_ABBREVIATION']
    home_merge = df.merge(
        odds[['game_key_home', 'whos_favored', 'spread', 'total']].rename(columns={'game_key_home': 'game_key'}),
        on='game_key',
        how='left',
        suffixes=('', '_home')
    )
    away_merge = df.merge(
        odds[['game_key_away', 'whos_favored', 'spread', 'total']].rename(columns={'game_key_away': 'game_key'}),
        on='game_key', 
        how='left',
        suffixes=('', '_away')
    )
    df['whos_favored'] = home_merge['whos_favored'].fillna(away_merge['whos_favored'])
    df['spread'] = home_merge['spread'].fillna(away_merge['spread']).round(2)
    df['total'] = home_merge['total'].fillna(away_merge['total']).round(2)
    df['team_is_favored'] = ((df['whos_favored'] == 'home') & (df['HOME_GAME'] == 1)) | \
                           ((df['whos_favored'] == 'away') & (df['HOME_GAME'] == 0))
    df['team_spread'] = df.apply(lambda row: 
        round(row['spread'] if row['HOME_GAME'] == 1 else -row['spread'], 2), axis=1)
    df.drop('game_key', axis=1, inplace=True)
    return df

team_dict = {
    'min': 'MIN', 
    'bos': 'BOS', 
    'bkn': 'BKN', 
    'ny': 'NYK', 
    'phi': 'PHI', 
    'tor': 'TOR', 
    'chi': 'CHI', 
    'cle': 'CLE', 
    'det': 'DET', 
    'ind': 'IND', 
    'mia': 'MIA', 
    'atl': 'ATL', 
    'cha': 'CHA', 
    'was': 'WAS',
    'wsh': 'WAS',
    'orl': 'ORL', 
    'mil': 'MIL', 
    'chh': 'CHH', 
    'dal': 'DAL', 
    'hou': 'HOU',
    'lac': 'LAC',
    'lal': 'LAL',
    'sac': 'SAC',
    'por': 'POR',
    'uta': 'UTA',
    'utah': 'UTA', 
    'den': 'DEN',
    'okc': 'OKC',
    'mem': 'MEM',
    'no': 'NOP',
    'sa': 'SAS',    
    'gs': 'GSW',
    'phx': 'PHX',  
}

In [10]:
bettingData = pd.read_csv('../DATA/CSV_FILES/bettingData.csv')
df = merge_betting_data(s20_regular, bettingData, team_dict)
df

,Unnamed: 0.1,Unnamed: 0,PLAYER_NAME,PLAYER_ID,MATCHUP,TEAM_ABBREVIATION,TEAM_ID,OPP_ABBREVIATION,HOME_GAME,GAME_ID,GAME_DATE,WL,PTS,AST,REB,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,STL,BLK,TOV,PLUS_MINUS,FANTASY_PTS,POINT_PER_SHOT,EFG,START_POSITION,COMMENT,OFF_RATING,E_OFF_RATING,DEF_RATING,E_DEF_RATING,NET_RATING,OREB_PCT,DREB_PCT,REB_PCT,AST_PCT,EFG_PCT,AST_TOV,USG_PCT,TS_PCT,E_PACE,PACE,PIE,POSS,PACE_PER40,E_USG_PCT,MIN,SPD,DIST,ORBC,DRBC,RBC,TCHS,SAST,FTAST,PASS,CFGM,CFGA,CFG_PCT,UFGM,UFGA,UFG_PCT,DFGM,DFGA,DFG_PCT,PTS_OFF_TOV,PTS_2ND_CHANCE,PTS_FB,PTS_PAINT,OPP_PTS_OFF_TOV,OPP_PTS_2ND_CHANCE,OPP_PTS_FB,OPP_PTS_PAINT,BLKA,PF,PFD,IS_PLAYOFF,TEAM_SEASON_ID,TEAM_NAME,TEAM_GAME_DATE,TEAM_MATCHUP,TEAM_WL,TEAM_MIN,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_STL,TEAM_BLK,TEAM_TOV,TEAM_PF,TEAM_PTS,TEAM_PLUS_MINUS,VIDEO_AVAILABLE,TEAM_PACE,GAME_PACE,OPP_PACE,OPP_TEAM_ID,TEAM_OFF_RATING,TEAM_DEF_RATING,OPP_DEF_RATING,OPP_OFF_RATING,OPP_PTS,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_REB,OPP_AST,OPP_STL,OPP_BLK,OPP_TOV,whos_favored,spread,total,team_is_favored,team_spread
0,0,0,Derrick Favors,202324,NOP @ TOR,NOP,1610612740,TOR,0,21900001,2019-10-22,L,6,2,7,3,6,0.500,0,0,NaN,0,0,NaN,1,6,0,1,1,-12,19.4,1.000,0.500000,C,NaN,108.2,108.2,132.7,132.7,-24.5,0.048,0.261,0.159,0.118,0.500,2.0,0.137,0.500,113.31,113.31,0.056,49,94.43,0.137,20.75,4.60,1.75,4,8,10,43,0,0,35,3,3,1.000,0,3,0.000,7,8,0.875,0.0,2.0,0.0,6.0,12.0,11.0,7.0,32.0,0.0,5.0,0.0,0,22019,New Orleans Pelicans,2019-10-22,NOP @ TOR,L,265,43,102,0.422,19,45,0.422,17,20,0.850,16,37,53,30,4,9,19,34,122,-8,1,106.2,106.2,106.2,1610612761,107.2,114.2,101.1,107.7,130.0,42.0,103.0,0.408,57.0,23.0,7.0,3.0,17.0,home,6.5,229.5,False,-6.5
1,1,1,Brandon Ingram,1627742,NOP @ TOR,NOP,1610612740,TOR,0,21900001,2019-10-22,L,22,5,5,8,19,0.421,2,5,0.400,4,4,1.0,0,5,1,2,2,-19,42.5,1.060,0.473684,F,NaN,102.6,102.6,122.5,122.5,-19.9,0.000,0.125,0.068,0.227,0.474,2.5,0.272,0.530,107.35,107.35,0.112,77,89.46,0.272,35.10,4.33,2.78,1,9,9,69,2,0,46,5,9,0.556,3,10,0.300,4,8,0.500,3.0,0.0,3.0,8.0,16.0,14.0,14.0,44.0,0.0,4.0,6.0,0,22019,New Orleans Pelicans,2019-10-22,NOP @ TOR,L,265,43,102,0.422,19,45,0.422,17,20,0.850,16,37,53,30,4,9,19,34,122,-8,1,106.2,106.2,106.2,1610612761,107.2,114.2,101.1,107.7,130.0,42.0,103.0,0.408,57.0,23.0,7.0,3.0,17.0,home,6.5,229.5,False,-6.5
2,2,2,Josh Hart,1628404,NOP @ TOR,NOP,1610612740,TOR,0,21900001,2019-10-22,L,15,1,10,4,9,0.444,3,5,0.600,4,4,1.0,4,6,0,1,1,-1,30.5,1.394,0.611111,NaN,NaN,105.5,105.5,101.7,101.7,3.7,0.111,0.167,0.139,0.067,0.611,1.0,0.174,0.697,96.28,96.28,0.213,55,80.24,0.174,28.17,4.31,2.22,5,8,13,42,0,0,27,2,5,0.400,2,4,0.500,2,4,0.500,0.0,4.0,2.0,2.0,10.0,5.0,12.0,22.0,1.0,4.0,4.0,0,22019,New Orleans Pelicans,2019-10-22,NOP @ TOR,L,265,43,102,0.422,19,45,0.422,17,20,0.850,16,37,53,30,4,9,19,34,122,-8,1,106.2,106.2,106.2,1610612761,107.2,114.2,101.1,107.7,130.0,42.0,103.0,0.408,57.0,23.0,7.0,3.0,17.0,home,6.5,229.5,False,-6.5
3,3,3,Lou Williams,101150,LAC vs. LAL,LAC,1610612746,LAL,1,21900002,2019-10-22,W,21,7,5,8,14,0.571,1,4,0.250,4,4,1.0,1,4,1,0,2,13,38.5,1.332,0.607143,NaN,NaN,121.6,121.6,102.7,102.7,19.0,0.029,0.098,0.066,0.280,0.607,3.5,0.214,0.666,97.38,97.38,0.195,74,81.15,0.214,36.72,3.76,2.46,1,5,6,89,0,2,62,1,2,0.500,7,12,0.583,0,1,0.000,8.0,2.0,5.0,6.0,9.0,6.0,5.0,30.0,1.0,0.0,9.0,0,22019,LA Clippers,2019-10-22,LAC vs. LAL,W,240,42,81,0.519,11,31,0.355,17,24,0.708,11,34,45,24,8,5,14,25,112,10,1,97.4,97.4,97.4,1610612747,118.4,107.9,111.7,101.8,102.0,37.0,85.0,0.435,41.0,20.0,4.0,7.0,15.0,away,3.5,224.0,False,3.5
4,4,4,Patrick Beverley,201976,LAC vs. LAL,LAC,1610612746,LAL,1,21900002,2019-10-22,W,2,6,10,1,7,0.143,0,5,0.000,0,0,NaN,2,8,0,1,2,13,24.0,0.286,0.142857,G,NaN,120.0,120.0,100.0,100.0,20.0,0.074,0.276,0.179,0.207,0.143,3.0,0.122,0.143,99.51,99.51,0.048,65,82.93,0.122,31.35,4.24,2.37,4,11,13,62,2,0,51,1,2,0.500,0,5,0.000,1,2,0.500,0.0,0.0